In [1]:
import os
from PIL import Image
from torchvision.transforms import v2
import torch
from torchvision.transforms import InterpolationMode
from tqdm import tqdm

In [2]:
def preprocess_cityscapes(root='./CityScapes/data', 
                         output_root='./preprocessed_data',
                         size=(512, 512),
                         preprocess_labels=True):
    """
    Preprocess Cityscapes dataset and save in the same structure.

    Args:
        root: Original dataset root directory
        output_root: Output directory for preprocessed data
        size: Target size for images (height, width)
        preprocess_labels: Whether to also preprocess label images
    """

    # Define transforms
    image_transform = v2.Compose([
        v2.Resize(size=size, interpolation=InterpolationMode.BILINEAR),
        v2.ToImage(), 
        v2.ToDtype(torch.float32, scale=True), 
    ])

    label_transform = v2.Compose([
        v2.Resize(size=size, interpolation=InterpolationMode.NEAREST),
        v2.PILToTensor(),
        v2.ToDtype(torch.int64, scale=False),
        v2.Lambda(lambda x: x.squeeze(0)) 
    ])

    # Define paths
    images_dir = os.path.join(root, "leftImg8bit_trainextra", "leftImg8bit", "train_extra")
    targets_dir = os.path.join(root, "gtCoarse", "train")

    output_images_dir = os.path.join(output_root, "leftImg8bit_trainextra", "leftImg8bit", "train_extra")
    output_targets_dir = os.path.join(output_root, "gtCoarse", "train")

    # Collect all image paths
    image_paths = []
    for city in os.listdir(images_dir):
        city_img_dir = os.path.join(images_dir, city)
        if not os.path.isdir(city_img_dir):
            continue
        for file in os.listdir(city_img_dir):
            if file.endswith("_leftImg8bit.png"):
                image_paths.append(os.path.join(city_img_dir, file))

    image_paths.sort()

    print(f"Found {len(image_paths)} images to preprocess")
    print(f"Target size: {size}")
    print(f"Output directory: {output_root}")
    print(f"Preprocessing labels: {preprocess_labels}")
    print()

    # Process images
    for img_path in tqdm(image_paths, desc="Preprocessing images"):
        # Read image
        image = Image.open(img_path).convert("RGB")

        # Apply transform
        image_tensor = image_transform(image)

        # Determine output path (preserve directory structure)
        rel_path = os.path.relpath(img_path, images_dir)
        output_img_path = os.path.join(output_images_dir, rel_path)
        output_img_path = output_img_path.replace("_leftImg8bit.png", "_leftImg8bit.pt")

        # Create output directory
        os.makedirs(os.path.dirname(output_img_path), exist_ok=True)

        # Save preprocessed image as tensor
        torch.save(image_tensor, output_img_path)

        # Process corresponding label if requested
        if preprocess_labels:
            target_path = (
                img_path
                .replace("leftImg8bit_trainextra/leftImg8bit", "gtCoarse")
                .replace("_leftImg8bit.png", "_gtCoarse_labelIds.png")
            )

            if os.path.exists(target_path):
                # Read label
                target = Image.open(target_path)

                # Apply transform
                target_tensor = label_transform(target)

                # Determine output path
                rel_target_path = os.path.relpath(target_path, targets_dir)
                output_target_path = os.path.join(output_targets_dir, rel_target_path)
                output_target_path = output_target_path.replace("_gtCoarse_labelIds.png", "_gtCoarse_labelIds.pt")

                # Create output directory
                os.makedirs(os.path.dirname(output_target_path), exist_ok=True)

                # Save preprocessed label
                torch.save(target_tensor, output_target_path)

    print(f"\nPreprocessing complete!")
    print(f"Preprocessed data saved to: {output_root}")


In [ ]:

# Run preprocessing
preprocess_cityscapes(
    root='./data/original_data',
    output_root='./data/preprocessed_data',
    size=(512, 512),
    preprocess_labels=True  # Set to False if you only need images
)


Found 19998 images to preprocess
Target size: (512, 512)
Output directory: ./preprocessed_data
Preprocessing labels: True



Preprocessing images: 100%|██████████| 19998/19998 [29:20<00:00, 11.36it/s]


Preprocessing complete!
Preprocessed data saved to: ./preprocessed_data
